# 第 2 章: iris データの探索と可視化

欠損値の分布と、品種ごとの特徴量の違いを確かめる。

Polyglot Notebooks（.NET Interactive）は 2026 年に廃止された。この Notebook は `Microsoft.dotnet-interactive` 1.0.712001 と Plotly.NET.Interactive 5.0.0 で動作を確かめている。
先に `dotnet build` で `apps/fsharp/` のライブラリをビルドしておく。

In [ ]:
#r "nuget: FSharp.Data, 8.2.0"
#r "nuget: Plotly.NET, 5.1.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "../src/MachineLearning/bin/Debug/net10.0/MachineLearning.dll"

In [ ]:
open System.IO
open Plotly.NET
open MachineLearning.Dataset
open MachineLearning.Chapter02.IrisPreprocessing

let irisCsv = Path.Combine(dataDir (), "iris.csv")
let rows = loadIris irisCsv
rows.Length

In [ ]:
let missing = rows |> List.map (fun row -> row.Features) |> countMissing

Chart.Column(values = (FeatureNames |> List.map (fun column -> missing[column])), Keys = FeatureNames)
|> Chart.withTitle "列ごとの欠損値の数"
|> Chart.withYAxisStyle "欠損値の数"

In [ ]:
rows
|> List.countBy (fun row -> row.Species)
|> List.map (fun (species, count) -> {| 種類 = species; 件数 = count |})
|> List.toArray

In [ ]:
let split = prepareIris irisCsv 0.3 0
let train = List.zip split.XTrain split.TTrain

train
|> List.groupBy snd
|> List.map (fun (species, group) ->
    Chart.Point(
        x = (group |> List.map (fun (features, _) -> features["花弁長さ"])),
        y = (group |> List.map (fun (features, _) -> features["花弁幅"])),
        Name = species
    ))
|> Chart.combine
|> Chart.withTitle "訓練データの花弁の長さと幅"
|> Chart.withXAxisStyle "花弁長さ"
|> Chart.withYAxisStyle "花弁幅"

In [ ]:
let meanOf column group =
    group |> List.averageBy (fun (features: Map<string, float>, _) -> features[column])

train
|> List.groupBy snd
|> List.sortBy fst
|> List.map (fun (species, group) ->
    {|
        種類 = species
        がく片長さ = meanOf "がく片長さ" group
        がく片幅 = meanOf "がく片幅" group
        花弁長さ = meanOf "花弁長さ" group
        花弁幅 = meanOf "花弁幅" group
    |})
|> List.toArray